In [1]:
from bs4 import BeautifulSoup
import time
import pandas as pd
import re

In [2]:
soup_list = []

for i in range(4):
    with open(f"mte_questions_filtered/mte_questions{i+1}.html", "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")
        soup_list.append(soup)

In [3]:
def get_exams_name(soup_list):
    exams = []
    years = []
    for soup in soup_list:
        for prova in soup.find_all("div", class_="q-question-info"):
            text = re.sub(r'\s+', ' ', prova.get_text()).strip()
            ano = text.split("Banca")[0].replace("Ano:", "")
            years.append(ano)
            exams.append(text.split("Prova:")[1].replace("|",""))
    return years, exams

In [4]:
years, provas = get_exams_name(soup_list)

In [7]:
def get_questions(soup_list):
    questions = []
    for soup in soup_list:
        for question in soup.find_all("div", class_="q-question-enunciation"):
            questions.append(re.sub(r'\s+', ' ', question.get_text()).strip())
    return questions

In [8]:
questions = get_questions(soup_list)

In [9]:
def get_options(soup_list):
    options = []
    for soup in soup_list:
        for option in soup.find_all("div", class_="q-item-enum js-alternative-content"):
            options.append(re.sub(r'\s+', ' ', option.get_text()).strip())
    return options

In [10]:
options = get_options(soup_list)

In [11]:
questions_json = []
for i in range(len(questions)):
    dict_question = {"question_number": i+1, "question": questions[i], "options": options[i*5:(i+1)*5], "exam": provas[i], "answer":""} 
    questions_json.append(dict_question)

In [12]:
df_questions = pd.DataFrame(questions_json)

In [14]:
def fix_space_punctuation(texto):
    return re.sub(r'([.!?,;:])([A-ZÀ-Ú])', r'\1 \2', texto)

def fix_options(options):
    fixed_options = []
    for option in options:
        fixed_option = option.replace('\n', ' ')
        fixed_options.append(fixed_option)
    return fixed_options

In [16]:
df_questions['question'] = df_questions['question'].str.replace('\n',' ')
df_questions['options'] = df_questions['options'].apply(fix_options)
df_questions['question'] = df_questions['question'].apply(fix_space_punctuation)
df_questions['question'] = df_questions['question'].str.replace('  ', ' ')

In [19]:
df_questions.to_csv('csvs/mte/mte_formacao_questions_unified_no_answer.csv', index=False, encoding='utf-8-sig')